In [ ]:
from pytorch_grad_cam import GradCAM, AblationCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image
import numpy as np
import torch
from torchvision.transforms import Compose, Normalize, ToTensor
import torchvision.models as models
import os
import time
from PIL import Image
from torchvision import transforms
import torch.nn as nn
from torchvision.datasets import ImageFolder
import matplotlib.pyplot as plt
import torch.nn.functional as F # 소프트맥스 활성화를 위해 추가


### 가중치 학습된 모델 불러오기

In [ ]:
weights = torch.load("model/lr1e4_512best_efficientB0_model_pretrained_weights827.pth", map_location=torch.device('cpu'))
model = models.efficientnet_b0(pretrained=False)
model.classifier[1] = nn.Linear(in_features=1280, out_features=13, bias=True)
model.load_state_dict(weights)
model

### 풀 데이터 셋 가져오기

In [ ]:
full_datasets = ImageFolder(root='../datasets/data')
class_names = full_datasets.classes
class_names

#### 데이터 전처리 코드 가져오기

In [ ]:
transforms_train = transforms.Compose([
    #transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

transforms_test = transforms.Compose([
    #transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

#### 데이터 학습시켰을때랑 똑같은 random 값 줘서 쪼개기

In [ ]:
from sklearn.model_selection import train_test_split
# 데이터 크기와 target 저장 
total_size = len(full_datasets)
targets = full_datasets.targets

train_indices, test_indices = train_test_split(
    np.arange(total_size),
    test_size=0.2,
    stratify=targets,
    random_state=42   
)

train_dataset = torch.utils.data.Subset(full_datasets, train_indices)
test_dataset = torch.utils.data.Subset(full_datasets, test_indices)

print("\n데이터셋 분할 완료.")
print(f"총 데이터 수: {total_size}")
print(f"학습용 데이터 수: {len(train_dataset)}")
print(f"검증용 데이터 수: {len(test_dataset)}")

전처리 적용

In [ ]:
test_dataset.dataset.transform = transforms_test

In [ ]:
img, label = test_dataset[0]

print(img.shape)
print(label)
print(len(test_dataset))

#### 그래드 캠에 쓸 타겟 레이어 설정

In [ ]:
target_layer = model.features[-1][0]
target_layer

#### 그래드캠 정의

In [ ]:
cam = GradCAM(model=model, target_layers=[target_layer])
cam.batch_size = 1

#### 원본 이미지 제대로 보기위해 역정규화

In [ ]:
# ImageNet 표준 정규화 값 (모델 학습 시 사용한 값과 일치해야 함)
# 만약 다른 값으로 정규화했다면 이 값을 변경해야 합니다.
NORM_MEAN = [0.485, 0.456, 0.406]
NORM_STD = [0.229, 0.224, 0.225]

def denormalize_image(tensor_image, mean=NORM_MEAN, std=NORM_STD):
    """
    정규화된 이미지 텐서를 역정규화하여 0-1 범위의 NumPy 배열로 변환합니다.
    """
    for t, m, s in zip(tensor_image, mean, std):
        t.mul_(s).add_(m) # t = t * s + m
    
    # 텐서를 NumPy 배열로 변환하고 채널 순서를 (H, W, C)로 변경
    np_image = tensor_image.numpy().transpose((1, 2, 0))
    
    # 픽셀 값을 0-1 범위로 클리핑하여 유효한 이미지 범위 유지
    np_image = np.clip(np_image, 0, 1)
    
    return np_image


### 갈색 유리병 특징맵이 잘 찾는지

In [ ]:
# 라벨 4인 이미지를 저장할 리스트
label_4_images_info = []
count = 0

# test_dataset에서 라벨이 4인 이미지 최대 100개를 찾습니다.
for i, (img, label) in enumerate(test_dataset):
    if label == 4:
        label_4_images_info.append((img, label, i)) # (이미지 텐서, 실제 라벨, 원본 인덱스)
        count += 1
        if count >= 100:
            break

In [ ]:
# 모델을 평가 모드로 설정
model.eval()

if not label_4_images_info:
    print("라벨이 4인 이미지를 찾을 수 없습니다.")
else:
    print(f"라벨이 4인 이미지 {len(label_4_images_info)}개를 찾았습니다. 시각화를 시작합니다.")

    # 각 이미지에 대해 Grad-CAM과 예측을 시각화
    for k, (original_img_tensor, actual_label, original_idx) in enumerate(label_4_images_info):
        # Grad-CAM을 적용하기 위해 이미지 차원을 확장합니다 (배치 크기 1).
        input_tensor = original_img_tensor.unsqueeze(0)
        
        # 모델 예측 수행
        with torch.no_grad():
            output = model(input_tensor)
            probabilities = F.softmax(output, dim=1) # 소프트맥스로 확률 변환
            predicted_prob, predicted_label = torch.max(probabilities, 1) # 가장 높은 확률과 해당 라벨
        
        # Grad-CAM 인스턴스에 적용할 타겟을 지정합니다. 여기서는 라벨 4에 대한 CAM을 계산합니다.
        # 실제 예측된 라벨에 대한 CAM을 보고 싶다면 targets = [ClassifierOutputTarget(predicted_label.item())] 로 변경
        targets = [ClassifierOutputTarget(4)] 
        
        # Grad-CAM을 계산합니다.
        grayscale_cam = cam(input_tensor=input_tensor, targets=targets)
        
        # 결과를 이미지에 덧입히기 위해 CAM 값을 첫 번째 차원을 제거합니다.
        grayscale_cam = grayscale_cam[0, :]

        # 원본 이미지 텐서를 역정규화하고 NumPy 배열로 변환
        denormalized_rgb_img = denormalize_image(original_img_tensor.cpu())
        
        # CAM 히트맵을 역정규화된 원본 이미지에 덧씌웁니다.
        visualization = show_cam_on_image(denormalized_rgb_img, grayscale_cam, use_rgb=True)

        # 결과 시각화
        plt.figure(figsize=(8, 4)) # 이미지 크기 조정
        
        plt.subplot(1, 2, 1)
        plt.imshow(denormalized_rgb_img)
        plt.title(f"Original Image (Actual: {actual_label})")
        plt.axis('off')

        plt.subplot(1, 2, 2)
        plt.imshow(visualization)
        plt.title(f"Grad-CAM (Predicted: {predicted_label.item()}, Prob: {predicted_prob.item():.2f})")
        plt.axis('off')
        
        plt.suptitle(f"Image {k+1}/{len(label_4_images_info)} (Dataset Index: {original_idx})")
        plt.show()
        
        # 3장만 우선 보고 싶다면, 여기서 break
        if k >= 100: 
           break

### 페트병 유색단일 Recall 낮은 순위 Top2

In [ ]:
# 라벨 페트병 유색단일(9) 이미지를 저장할 리스트
label_9_images_info = []
count = 0

# test_dataset에서 라벨이 4인 이미지 최대 100개를 찾습니다.
for i, (img, label) in enumerate(test_dataset):
    if label == 9:
        label_9_images_info.append((img, label, i)) # (이미지 텐서, 실제 라벨, 원본 인덱스)
        #count += 1
        # if count >= 100:
        #     break

In [ ]:
# 모델을 평가 모드로 설정
model.eval()

if not label_9_images_info:
    print("라벨이 9인 이미지를 찾을 수 없습니다.")
else:
    print(f"라벨이 9인 이미지 {len(label_9_images_info)}개를 찾았습니다. 시각화를 시작합니다.")

    # 각 이미지에 대해 Grad-CAM과 예측을 시각화
    for k, (original_img_tensor, actual_label, original_idx) in enumerate(label_9_images_info):
        # Grad-CAM을 적용하기 위해 이미지 차원을 확장합니다 (배치 크기 1).
        input_tensor = original_img_tensor.unsqueeze(0)
        
        # 모델 예측 수행
        with torch.no_grad():
            output = model(input_tensor)
            probabilities = F.softmax(output, dim=1) # 소프트맥스로 확률 변환
            predicted_prob, predicted_label = torch.max(probabilities, 1) # 가장 높은 확률과 해당 라벨
        
        if (predicted_label != 9):
            # Grad-CAM 인스턴스에 적용할 타겟을 지정합니다. 여기서는 라벨 9에 대한 CAM을 계산합니다.
            # 실제 예측된 라벨에 대한 CAM을 보고 싶다면 targets = [ClassifierOutputTarget(predicted_label.item())] 로 변경
            targets = [ClassifierOutputTarget(9)] 
            
            # Grad-CAM을 계산합니다.
            grayscale_cam = cam(input_tensor=input_tensor, targets=targets)
            
            # 결과를 이미지에 덧입히기 위해 CAM 값을 첫 번째 차원을 제거합니다.
            grayscale_cam = grayscale_cam[0, :]

            # 원본 이미지 텐서를 역정규화하고 NumPy 배열로 변환
            denormalized_rgb_img = denormalize_image(original_img_tensor.cpu())
            
            # CAM 히트맵을 역정규화된 원본 이미지에 덧씌웁니다.
            visualization = show_cam_on_image(denormalized_rgb_img, grayscale_cam, use_rgb=True)

            # 결과 시각화
            plt.figure(figsize=(8, 4)) # 이미지 크기 조정
            
            plt.subplot(1, 2, 1)
            plt.imshow(denormalized_rgb_img)
            plt.title(f"Original Image (Actual: {actual_label})")
            plt.axis('off')

            plt.subplot(1, 2, 2)
            plt.imshow(visualization)
            plt.title(f"Grad-CAM (Predicted: {predicted_label.item()}, Prob: {predicted_prob.item():.2f})")
            plt.axis('off')
            
            plt.suptitle(f"Image {k+1}/{len(label_9_images_info)} (Dataset Index: {original_idx})")
            plt.show()
            
            # 3장만 우선 보고 싶다면, 여기서 break
            # if k >= 100: 
            #    break

### 플라스틱PE Recall 낮은 순위 Top3

In [ ]:
# 라벨 플라스틱PE(10) 이미지를 저장할 리스트
label_10_images_info = []
count = 0

# test_dataset에서 라벨이 10인 이미지 최대 100개를 찾습니다.
for i, (img, label) in enumerate(test_dataset):
    if label == 10:
        label_10_images_info.append((img, label, i)) # (이미지 텐서, 실제 라벨, 원본 인덱스)
        # count += 1
        # if count >= 100:
        #     break

In [ ]:
# 모델을 평가 모드로 설정
model.eval()
error_cnt10 = 0
if not label_10_images_info:
    print("라벨이 10인 이미지를 찾을 수 없습니다.")
else:
    print(f"라벨이 10인 이미지 {len(label_10_images_info)}개를 찾았습니다. 시각화를 시작합니다.")

    # 각 이미지에 대해 Grad-CAM과 예측을 시각화
    for k, (original_img_tensor, actual_label, original_idx) in enumerate(label_10_images_info):
        # Grad-CAM을 적용하기 위해 이미지 차원을 확장합니다 (배치 크기 1).
        input_tensor = original_img_tensor.unsqueeze(0)
        
        # 모델 예측 수행
        with torch.no_grad():
            output = model(input_tensor)
            probabilities = F.softmax(output, dim=1) # 소프트맥스로 확률 변환
            predicted_prob, predicted_label = torch.max(probabilities, 1) # 가장 높은 확률과 해당 라벨
        
        if (predicted_label != 10):
            error_cnt10 += 1
            # Grad-CAM 인스턴스에 적용할 타겟을 지정합니다. 여기서는 라벨 10에 대한 CAM을 계산합니다.
            # 실제 예측된 라벨에 대한 CAM을 보고 싶다면 targets = [ClassifierOutputTarget(predicted_label.item())] 로 변경
            targets = [ClassifierOutputTarget(10)] 
            
            # Grad-CAM을 계산합니다.
            grayscale_cam = cam(input_tensor=input_tensor, targets=targets)
            
            # 결과를 이미지에 덧입히기 위해 CAM 값을 첫 번째 차원을 제거합니다.
            grayscale_cam = grayscale_cam[0, :]

            # 원본 이미지 텐서를 역정규화하고 NumPy 배열로 변환
            denormalized_rgb_img = denormalize_image(original_img_tensor.cpu())
            
            # CAM 히트맵을 역정규화된 원본 이미지에 덧씌웁니다.
            visualization = show_cam_on_image(denormalized_rgb_img, grayscale_cam, use_rgb=True)

            # 결과 시각화
            plt.figure(figsize=(8, 4)) # 이미지 크기 조정
            
            plt.subplot(1, 2, 1)
            plt.imshow(denormalized_rgb_img)
            plt.title(f"Original Image (Actual: {actual_label})")
            plt.axis('off')

            plt.subplot(1, 2, 2)
            plt.imshow(visualization)
            plt.title(f"Grad-CAM (Predicted: {predicted_label.item()}, Prob: {predicted_prob.item():.2f})")
            plt.axis('off')
            
            plt.suptitle(f"Image {k+1}/{len(label_10_images_info)} (Dataset Index: {original_idx})")
            plt.show()
            
            # 3장만 우선 보고 싶다면, 여기서 break
            if k >= 100: 
               break
print(error_cnt10)

### 플라스틱PP Recall 낮은 순위 Top4

In [ ]:
# 라벨 플라스틱PP(11) 이미지를 저장할 리스트
label_11_images_info = []
count = 0

# test_dataset에서 라벨이 11인 이미지 최대 100개를 찾습니다.
for i, (img, label) in enumerate(test_dataset):
    if label == 11:
        label_11_images_info.append((img, label, i)) # (이미지 텐서, 실제 라벨, 원본 인덱스)
        # count += 1
        # if count >= 100:
        #     break

In [ ]:
# 모델을 평가 모드로 설정
model.eval()
error_cnt11 = 0

if not label_11_images_info:
    print("라벨이 11인 이미지를 찾을 수 없습니다.")
else:
    print(f"라벨이 11인 이미지 {len(label_11_images_info)}개를 찾았습니다. 시각화를 시작합니다.")

    # 각 이미지에 대해 Grad-CAM과 예측을 시각화
    for k, (original_img_tensor, actual_label, original_idx) in enumerate(label_11_images_info):
        # Grad-CAM을 적용하기 위해 이미지 차원을 확장합니다 (배치 크기 1).
        input_tensor = original_img_tensor.unsqueeze(0)
        
        # 모델 예측 수행
        with torch.no_grad():
            output = model(input_tensor)
            probabilities = F.softmax(output, dim=1) # 소프트맥스로 확률 변환
            predicted_prob, predicted_label = torch.max(probabilities, 1) # 가장 높은 확률과 해당 라벨
        
        if predicted_label != 11:
            error_cnt11 += 1
            # Grad-CAM 인스턴스에 적용할 타겟을 지정합니다. 여기서는 라벨 1에 대한 CAM을 계산합니다.
            # 실제 예측된 라벨에 대한 CAM을 보고 싶다면 targets = [ClassifierOutputTarget(predicted_label.item())] 로 변경
            targets = [ClassifierOutputTarget(11)] 
            
            # Grad-CAM을 계산합니다.
            grayscale_cam = cam(input_tensor=input_tensor, targets=targets)
            
            # 결과를 이미지에 덧입히기 위해 CAM 값을 첫 번째 차원을 제거합니다.
            grayscale_cam = grayscale_cam[0, :]

            # 원본 이미지 텐서를 역정규화하고 NumPy 배열로 변환
            denormalized_rgb_img = denormalize_image(original_img_tensor.cpu())
            
            # CAM 히트맵을 역정규화된 원본 이미지에 덧씌웁니다.
            visualization = show_cam_on_image(denormalized_rgb_img, grayscale_cam, use_rgb=True)

            # 결과 시각화
            plt.figure(figsize=(8, 4)) # 이미지 크기 조정
            
            plt.subplot(1, 2, 1)
            plt.imshow(denormalized_rgb_img)
            plt.title(f"Original Image (Actual: {actual_label})")
            plt.axis('off')

            plt.subplot(1, 2, 2)
            plt.imshow(visualization)
            plt.title(f"Grad-CAM (Predicted: {predicted_label.item()}, Prob: {predicted_prob.item():.2f})")
            plt.axis('off')
            
            plt.suptitle(f"Image {k+1}/{len(label_11_images_info)} (Dataset Index: {original_idx})")
            plt.show()
            
            # # 3장만 우선 보고 싶다면, 여기서 break
            if k >= 100: 
                break
print(error_cnt11)

### 플라스틱PS Recall 낮은 순위 Top1

In [ ]:
# 라벨 플라스틱PS(12) 이미지를 저장할 리스트
label_12_images_info = []
count = 0

# test_dataset에서 라벨이 12인 이미지 최대 100개를 찾습니다.
for i, (img, label) in enumerate(test_dataset):
    if label == 12:
        label_12_images_info.append((img, label, i)) # (이미지 텐서, 실제 라벨, 원본 인덱스)
        # count += 1
        # if count >= 100:
        #     break

In [ ]:
# 모델을 평가 모드로 설정
model.eval()
error_cnt12 = 0

if not label_12_images_info:
    print("라벨이 12인 이미지를 찾을 수 없습니다.")
else:
    print(f"라벨이 12인 이미지 {len(label_12_images_info)}개를 찾았습니다. 시각화를 시작합니다.")

    # 각 이미지에 대해 Grad-CAM과 예측을 시각화
    for k, (original_img_tensor, actual_label, original_idx) in enumerate(label_12_images_info):
        # Grad-CAM을 적용하기 위해 이미지 차원을 확장합니다 (배치 크기 1).
        input_tensor = original_img_tensor.unsqueeze(0)
        
        # 모델 예측 수행
        with torch.no_grad():
            output = model(input_tensor)
            probabilities = F.softmax(output, dim=1) # 소프트맥스로 확률 변환
            predicted_prob, predicted_label = torch.max(probabilities, 1) # 가장 높은 확률과 해당 라벨
        
        if predicted_label != 12:
            error_cnt12 += 1
            # Grad-CAM 인스턴스에 적용할 타겟을 지정합니다. 여기서는 라벨 4에 대한 CAM을 계산합니다.
            # 실제 예측된 라벨에 대한 CAM을 보고 싶다면 targets = [ClassifierOutputTarget(predicted_label.item())] 로 변경
            targets = [ClassifierOutputTarget(12)] 
            
            # Grad-CAM을 계산합니다.
            grayscale_cam = cam(input_tensor=input_tensor, targets=targets)
            
            # 결과를 이미지에 덧입히기 위해 CAM 값을 첫 번째 차원을 제거합니다.
            grayscale_cam = grayscale_cam[0, :]

            # 원본 이미지 텐서를 역정규화하고 NumPy 배열로 변환
            denormalized_rgb_img = denormalize_image(original_img_tensor.cpu())
            
            # CAM 히트맵을 역정규화된 원본 이미지에 덧씌웁니다.
            visualization = show_cam_on_image(denormalized_rgb_img, grayscale_cam, use_rgb=True)

            # 결과 시각화
            plt.figure(figsize=(8, 4)) # 이미지 크기 조정
            
            plt.subplot(1, 2, 1)
            plt.imshow(denormalized_rgb_img)
            plt.title(f"Original Image (Actual: {actual_label})")
            plt.axis('off')

            plt.subplot(1, 2, 2)
            plt.imshow(visualization)
            plt.title(f"Grad-CAM (Predicted: {predicted_label.item()}, Prob: {predicted_prob.item():.2f})")
            plt.axis('off')
            
            plt.suptitle(f"Image {k+1}/{len(label_12_images_info)} (Dataset Index: {original_idx})")
            plt.show()
            
            # # 3장만 우선 보고 싶다면, 여기서 break
            if k >= 100: 
                break
print(error_cnt12)

### 페트병유색, 플라스틱PE, 플라스틱PP, 플라스틱PS 가 각각 어떤걸로 잘못 예측되고있는지 확인

In [ ]:
# 페트병유색
# 모델을 평가 모드로 설정
model.eval()
error_cnt9 = 0
error_label_9_list = [0] * 13

if not label_9_images_info:
    print("라벨이 9인 이미지를 찾을 수 없습니다.")
else:
    # 각 이미지에 대해 Grad-CAM과 예측을 시각화
    for k, (original_img_tensor, actual_label, original_idx) in enumerate(label_9_images_info):
        # Grad-CAM을 적용하기 위해 이미지 차원을 확장합니다 (배치 크기 1).
        input_tensor = original_img_tensor.unsqueeze(0)
        
        # 모델 예측 수행
        with torch.no_grad():
            output = model(input_tensor)
            probabilities = F.softmax(output, dim=1) # 소프트맥스로 확률 변환
            predicted_prob, predicted_label = torch.max(probabilities, 1) # 가장 높은 확률과 해당 라벨
        
        if predicted_label != 9:
            error_cnt9 += 1
            error_label_9_list[predicted_label] += 1



In [ ]:
print(F"페트병유색 틀린 개수: {error_cnt9}/{len(label_9_images_info)}")
for i in range(13):
    print(f"{class_names[i]}-{i} 레이블로 잘못 예측한 개수: {error_label_9_list[i]}")

In [ ]:
# 플라스틱 PE
# 모델을 평가 모드로 설정
model.eval()
error_cnt10 = 0
error_label_10_list = [0] * 13

if not label_10_images_info:
    print("라벨이 10인 이미지를 찾을 수 없습니다.")
else:
    # 각 이미지에 대해 Grad-CAM과 예측을 시각화
    for k, (original_img_tensor, actual_label, original_idx) in enumerate(label_10_images_info):
        # Grad-CAM을 적용하기 위해 이미지 차원을 확장합니다 (배치 크기 1).
        input_tensor = original_img_tensor.unsqueeze(0)
        
        # 모델 예측 수행
        with torch.no_grad():
            output = model(input_tensor)
            probabilities = F.softmax(output, dim=1) # 소프트맥스로 확률 변환
            predicted_prob, predicted_label = torch.max(probabilities, 1) # 가장 높은 확률과 해당 라벨
        
        if predicted_label != 10:
            error_cnt10 += 1
            error_label_10_list[predicted_label] += 1

print(F"플라스틱 PE 틀린 개수: {error_cnt10}/{len(label_10_images_info)}")
for i in range(13):
    print(f"{class_names[i]}-{i} 레이블로 잘못 예측한 개수: {error_label_10_list[i]}")

In [ ]:
# 플라스틱 PP
# 모델을 평가 모드로 설정
model.eval()
error_cnt11 = 0
error_label_11_list = [0] * 13

if not label_11_images_info:
    print("라벨이 11인 이미지를 찾을 수 없습니다.")
else:
    # 각 이미지에 대해 Grad-CAM과 예측을 시각화
    for k, (original_img_tensor, actual_label, original_idx) in enumerate(label_11_images_info):
        # Grad-CAM을 적용하기 위해 이미지 차원을 확장합니다 (배치 크기 1).
        input_tensor = original_img_tensor.unsqueeze(0)
        
        # 모델 예측 수행
        with torch.no_grad():
            output = model(input_tensor)
            probabilities = F.softmax(output, dim=1) # 소프트맥스로 확률 변환
            predicted_prob, predicted_label = torch.max(probabilities, 1) # 가장 높은 확률과 해당 라벨
        
        if predicted_label != 11:
            error_cnt11 += 1
            error_label_11_list[predicted_label] += 1

print(F"플라스틱 PP 틀린 개수: {error_cnt11}/{len(label_11_images_info)}")
for i in range(13):
    print(f"{class_names[i]}-{i} 레이블로 잘못 예측한 개수: {error_label_11_list[i]}")

In [ ]:
# 플라스틱 PS
# 모델을 평가 모드로 설정
model.eval()
error_cnt12 = 0
error_label_12_list = [0] * 13

if not label_12_images_info:
    print("라벨이 12인 이미지를 찾을 수 없습니다.")
else:
    # 각 이미지에 대해 Grad-CAM과 예측을 시각화
    for k, (original_img_tensor, actual_label, original_idx) in enumerate(label_12_images_info):
        # Grad-CAM을 적용하기 위해 이미지 차원을 확장합니다 (배치 크기 1).
        input_tensor = original_img_tensor.unsqueeze(0)
        
        # 모델 예측 수행
        with torch.no_grad():
            output = model(input_tensor)
            probabilities = F.softmax(output, dim=1) # 소프트맥스로 확률 변환
            predicted_prob, predicted_label = torch.max(probabilities, 1) # 가장 높은 확률과 해당 라벨
        
        if predicted_label != 12:
            error_cnt12 += 1
            error_label_12_list[predicted_label] += 1

print(F"플라스틱 PS 틀린 개수: {error_cnt12}/{len(label_12_images_info)}")
for i in range(13):
    print(f"{class_names[i]}-{i} 레이블로 잘못 예측한 개수: {error_label_12_list[i]}")


In [ ]:
# # 라벨 4인 첫 번째 이미지와 그 인덱스를 찾습니다.
# label_4_image = None
# label_4_index = -1
# for i, (img, label) in enumerate(test_dataset):
#     if label == 7:
#         label_4_image = img
#         label_4_index = i
#         break

# if label_4_image is None:
#     print("라벨이 4인 이미지를 찾을 수 없습니다.")
# else:
#     print(f"라벨 4인 이미지 발견! 인덱스: {label_4_index}")
    
#     # Grad-CAM을 적용하기 위해 이미지 차원을 확장합니다 (배치 크기 1).
#     input_tensor = label_4_image.unsqueeze(0)
    
#     # Grad-CAM 인스턴스에 적용할 타겟을 지정합니다. 여기서는 라벨 4에 대한 CAM을 계산합니다.
#     targets = [ClassifierOutputTarget(4)]
    
#     # Grad-CAM을 계산합니다.
#     grayscale_cam = cam(input_tensor=input_tensor, targets=targets)

#     # 결과를 이미지에 덧입히기 위해 CAM 값을 첫 번째 차원을 제거합니다.
#     grayscale_cam = grayscale_cam[0, :]

#     # 원본 이미지를 시각화를 위해 NumPy 배열로 변환합니다.
#     rgb_img = np.float32(label_4_image) / 255
#     rgb_img = np.transpose(rgb_img, (1,2,0))
    
#     # CAM 히트맵을 원본 이미지에 덧씌웁니다.
#     visualization = show_cam_on_image(rgb_img, grayscale_cam, use_rgb=True)

#     # 결과 시각화
#     plt.imshow(visualization)
#     plt.title(f"Grad-CAM Visualization for Label 4")
#     plt.axis('off')
#     plt.show()
